# Brut force des poids finaux

> **Note ajoutee apres le hackathon.**
> En relisant ce notebook je me suis rendu compte d'une erreur : `MODEL_NAMES` contient 3 noms
> alors que seulement 2 fichiers sont charges (catboost etait commente, je n'ai pas eu le temps
> de l'entrainer). `zip()` s'arrete au plus court, donc les predictions **LightGBM** sont
> affichees sous le nom **catboost** dans les sorties ci-dessous.
>
> Les poids et les scores sont justes, c'est uniquement l'etiquette qui est fausse :
> le `0.2659` attribue a catboost est en realite le poids de LightGBM.
>
> La version corrigee, avec une verification de la coherence des arguments, est dans `src/blend.py`.


In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error
import os

## 1. Configuration

In [ ]:
TARGETS = ['satisfaction', 'wip', 'investissement']
MODEL_NAMES = ['ft_transformer', 'catboost', 'lightgbm']
VAL_FILES = [
    'stacking_data/ft_transformer_val_preds.csv',
    #'stacking_data/catboost_val_preds.csv', 
    'stacking_data/lightgbm_val_preds.csv'
]
TEST_FILES = [
    'outputs/ft_transformer_L1Loss_Scheduler.csv',
    #'outputs/catboost_for_stacking.csv',
    'outputs/lightGBM_10000estim_top400features.csv'
]

## 2. Fonctions utilitaires

In [ ]:
def load_and_merge_preds(file_paths, model_names, is_test=False):
    """Fusionne les prédictions de plusieurs modèles en un seul DataFrame"""
    df_merged = None
    for filepath, name in zip(file_paths, model_names):
        df = pd.read_csv(filepath)
        if is_test:
            rename_dict = {t: f'pred_{t}_{name}' for t in TARGETS}
        else:
            rename_dict = {c: f"{c}_{name}" for c in df.columns if 'pred' in c}
        df = df.rename(columns=rename_dict)
        if df_merged is None:
            df_merged = df
        else:
            cols_to_use = ['id'] + list(rename_dict.values())
            df_merged = df_merged.merge(df[cols_to_use], on='id', how='left')
            
    return df_merged

## 3. Préparation des données du stacking

In [ ]:
df_meta_train = load_and_merge_preds(VAL_FILES, MODEL_NAMES, is_test=False)
for t in TARGETS:
    if f'true_{t}' not in df_meta_train.columns:
        raise ValueError(f"La colonne cible réelle 'true_{t}' est manquante dans les fichiers de validation.")
print(f"Shape Meta-Train: {df_meta_train.shape}")
df_meta_train.head()

Chargement et fusion des données de Validation...
Shape Meta-Train: (15381, 10)


,id,pred_satisfaction_ft_transformer,pred_wip_ft_transformer,pred_investissement_ft_transformer,true_satisfaction,true_wip,true_investissement,pred_satisfaction_catboost,pred_wip_catboost,pred_investissement_catboost
0,76828,0.939423,28914848.0,873859.44,0.953826,29168262.0,875000.0,0.957426,2.893948e+07,8.749039e+05
1,73881,0.719375,28912392.0,1251523.60,0.761158,29974464.0,1250000.0,0.736065,2.862848e+07,1.249998e+06
2,2009,0.877888,33779144.0,873972.10,0.846764,35647990.0,875000.0,0.822626,3.133251e+07,8.750041e+05
3,26610,0.632180,21881818.0,1129140.10,0.534519,20229534.0,1125000.0,0.682325,2.692735e+07,1.125017e+06
4,70500,0.955470,28020940.0,1000052.80,0.950798,28363276.0,1000000.0,0.946254,2.863267e+07,1.000002e+06


## 4. Entrainement du meta-modèle

In [ ]:
best_weights_global = {}
def vector_hit_rate(y_true, y_pred, delta=0.05):
    return np.mean(np.abs(y_true - y_pred) <= delta)
for target in TARGETS:
    print(f"\n> Optimisation pour : {target.upper()}")
    cols = [c for c in df_meta_train.columns if f'pred_{target}' in c]
    cols.sort() 
    X_mat = df_meta_train[cols].values
    y_true = df_meta_train[f'true_{target}'].values
    n_trials = 10000
    best_score = -1
    best_weights = None
    random_weights = np.random.dirichlet(np.ones(X_mat.shape[1]), size=n_trials)
    all_preds = np.dot(X_mat, random_weights.T)
    abs_errors = np.abs(all_preds - y_true[:, None])
    hit_rates = np.mean(abs_errors <= 0.05, axis=0)
    best_idx = np.argmax(hit_rates)
    best_score = hit_rates[best_idx]
    best_weights = random_weights[best_idx]
    weights_named = dict(zip(cols, best_weights))
    best_weights_global[target] = weights_named
    print(f"  Meilleur Hit Rate Validation : {best_score:.4%}")
    print(f"  Poids optimaux :")
    for col_name, w in weights_named.items():
        clean_name = col_name.replace(f'pred_{target}_', '')
        print(f"    - {clean_name}: {w:.4f}")


--- Recherche des Meilleurs Poids (Optimisation Hit Rate) ---

> Optimisation pour : SATISFACTION
  Meilleur Hit Rate Validation : 81.9583%
  Poids optimaux :
    - catboost: 0.2659
    - ft_transformer: 0.7341

> Optimisation pour : WIP
  Meilleur Hit Rate Validation : 0.0065%
  Poids optimaux :
    - catboost: 0.0387
    - ft_transformer: 0.9613

> Optimisation pour : INVESTISSEMENT
  Meilleur Hit Rate Validation : 0.6111%
  Poids optimaux :
    - catboost: 0.9992
    - ft_transformer: 0.0008


## 5. Prédiction finale

In [ ]:

df_meta_test = load_and_merge_preds(TEST_FILES, MODEL_NAMES, is_test=True)
final_submission = pd.DataFrame({'id': df_meta_test['id']})
for target in TARGETS:
    weights_dict = best_weights_global[target]
    final_pred = np.zeros(len(df_meta_test))
    for col_name, weight in weights_dict.items():
        if col_name in df_meta_test.columns:
            final_pred += df_meta_test[col_name].values * weight
        else:
            model_suffix = col_name.split('_')[-1] 
            actual_col = [c for c in df_meta_test.columns if target in c and model_suffix in c][0]
            final_pred += df_meta_test[actual_col].values * weight
    final_submission[target] = final_pred

## 6. Sauvegarde

In [13]:
os.makedirs('outputs', exist_ok=True)
output_path = 'outputs/submission_optimized_blender.csv'
final_submission.to_csv(output_path, index=False)
print(f"Soumission générée : {output_path}")

Soumission générée : outputs/submission_optimized_blender.csv
